In [1]:
import os
import glob
from structure_feature_extractor import (
    extract_motif_features,
    auto_detect_a_site_species,
    analyze_motif_connectivity,
    compute_three_layer_spatial_features,
    compute_motif_weights_standardized,
    weighted_features
)
from define_structural_primitives import (
get_crystalnn_motifs_adaptive,
identify_asymmetric_unit
)
from pymatgen.core import Structure
import numpy as np
from CE_feature import (
extract_CE_features_from_motifs,
rename_CE_feature_columns,
weighted_CE_vector
)

In [2]:
cif_folder = '/home/gaotianjiao/modnet-master/ext_struc/perovskite_cif'
cif_files = glob.glob(os.path.join(cif_folder, '*.cif'))
print(f"共找到 {len(cif_files)} 个 CIF 文件")

共找到 2240 个 CIF 文件


In [3]:
import pandas as pd
import numpy as np
from tqdm import tqdm  
import copy
# 结果容器
all_results = []
failed_files = []

# 使用 tqdm 显示进度条
for cif_file in tqdm(cif_files, desc="Processing CIFs"):
    try:
        # 1. 加载结构
        struct = Structure.from_file(cif_file)
        print(f"化学式: {struct.formula}")
        # 2. 识别不对称单元 (Asymmetric Unit)
        inequivalent_sites, equivalent_sites, space_group_info, symmetrized_structure = identify_asymmetric_unit(struct)
        
        structural_motifs,structure = get_crystalnn_motifs_adaptive(
            struct, 
            symmetrized_structure, 
            weight_threshold=0.7,  
            angle_tolerance=15
        )

        # 筛选代表性基元 (用于局部特征)
        representative_motifs = [
             m for m in structural_motifs
             if m['central_atom']['index'] == m['central_atom']['representative_index']
        ]
        ce_motifs = copy.deepcopy(structural_motifs)
        
        for m in ce_motifs:
            raw_center = m['central_atom']['species']
            m['central_atom']['species'] = "".join([c for c in raw_center if c.isalpha()])
            
            for neighbor in m['neighbors']:
                raw_neigh = neighbor['species']
                neighbor['species'] = "".join([c for c in raw_neigh if c.isalpha()])
        # --- A. CE (Chemical Environment) 特征 ---
        ce_features, property_names = extract_CE_features_from_motifs(
            ce_motifs, 
            "/home/gaotianjiao/modnet-master/ext_struc/element_properties/ElementsProperties.xlsx"
        )
        final_CE_vector = weighted_CE_vector(ce_features, method='hybrid')
        # 将 Series 转为 字典，方便后续合并
        if isinstance(final_CE_vector, (pd.Series, np.ndarray, list)):
             # 确保它是带列名的 Series，然后转 dict
             temp_df = rename_CE_feature_columns(pd.DataFrame([final_CE_vector]), property_names)
             ce_dict = temp_df.iloc[0].to_dict()
        else:
             ce_dict = final_CE_vector # 假设已经是dict

        # --- B. 局部基元特征 (Local Features) ---
        motif_features = extract_motif_features(representative_motifs)
        # 计算权重
        weights, all_keys, X_scaled, var_w, dist_w = compute_motif_weights_standardized(motif_features, method='hybrid')
        # 加权聚合
        local_features_dict = weighted_features(motif_features, weights)
       
        # --- C. 连接特征 (Connectivity) ---
        auto_ignore_list = auto_detect_a_site_species(structural_motifs, cn_threshold=8.0)
        connectivity_dict = analyze_motif_connectivity(structural_motifs, structure,ignore_species=auto_ignore_list,cation_only=True)
    
        # --- D. 空间排布特征 (Spatial Arrangement) ---
        arrangement_dict = compute_three_layer_spatial_features(structural_motifs,structure)

        single_entry = {
            'cif_file': os.path.basename(cif_file),
            'n_inequivalent': len(inequivalent_sites),
            'reduction_ratio': len(inequivalent_sites)/len(struct)
        }
        
        single_entry.update(ce_dict)            # 展开 CE 特征
        single_entry.update(local_features_dict) # 展开局部特征
        single_entry.update(connectivity_dict)   # 展开连接特征
        single_entry.update(arrangement_dict)    # 展开空间特征
        
        all_results.append(single_entry)

    except Exception as e:
        error_msg = f"{os.path.basename(cif_file)}: {str(e)}"
        failed_files.append(error_msg)

Processing CIFs:   0%|          | 0/2240 [00:00<?, ?it/s]

化学式: Ca1 Zr1 O3
-> [Step 0] 预处理: 原子数 5
-> [Step 2] Phase 1: 标准拓扑扫描...
-> [Step 3] Phase 2: 检查孤立原子 (强制半径 4.0 Å)...
-> [Step 4] Phase 3: 全局对称化...
-> [Step 5] 生成特征...


/home/gaotianjiao/modnet-master/ext_struc/structure_feature_extractor.py:790: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  f'{prefix}_skew': skew(values),  # 偏向 (Asymmetry)
/home/gaotianjiao/modnet-master/ext_struc/structure_feature_extractor.py:791: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  f'{prefix}_kurt': kurtosis(values)  # 集中度 (Peakedness)
Processing CIFs:   0%|          | 1/2240 [00:02<1:36:17,  2.58s/it]

------------------------------
[Auto-Detect] 正在侦测 A 位阳离子...
   -> 元素 Ca: 平均 CN = 12.00
判定为 A 位 (CN > 8.0) -> 加入忽略列表
   -> 元素 Zr: 平均 CN = 6.00
判定为骨架 B 位/其他 -> 保留
   -> 元素 O: 平均 CN = 6.00
判定为骨架 B 位/其他 -> 保留
自动生成的忽略列表: ['Ca']
------------------------------
 开始通用结构连接性分析
截断半径: 6.00 Å
忽略列表 (A位/杂质): ['Ca']
仅分析阳离子模式: True
识别到的桥连阴离子: ['O']
🔹 有效骨架节点数: 27 / 135 (已过滤 A位/阴离子)
🔹 正在提取图特征 (自动清洗孤立节点)...
✅ 分析完成
化学式: Ce1 Ti1 O3
-> [Step 0] 预处理: 原子数 5
-> [Step 2] Phase 1: 标准拓扑扫描...
-> [Step 3] Phase 2: 检查孤立原子 (强制半径 4.0 Å)...
-> [Step 4] Phase 3: 全局对称化...
-> [Step 5] 生成特征...


/home/gaotianjiao/modnet-master/ext_struc/structure_feature_extractor.py:790: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  f'{prefix}_skew': skew(values),  # 偏向 (Asymmetry)
/home/gaotianjiao/modnet-master/ext_struc/structure_feature_extractor.py:791: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  f'{prefix}_kurt': kurtosis(values)  # 集中度 (Peakedness)
Processing CIFs:   0%|          | 2/2240 [00:05<1:33:32,  2.51s/it]

...

-> [Step 3] Phase 2: 检查孤立原子 (强制半径 4.0 Å)...
-> [Step 4] Phase 3: 全局对称化...
-> [Step 5] 生成特征...
化学式: Rb8 Na4 Yb4 F24
-> [Step 0] 预处理: 原子数 40
-> [Step 2] Phase 1: 标准拓扑扫描...


Processing CIFs: 100%|█████████▉| 2239/2240 [45:29<00:01,  1.02s/it]

-> [Step 3] Phase 2: 检查孤立原子 (强制半径 4.0 Å)...
-> [Step 4] Phase 3: 全局对称化...
-> [Step 5] 生成特征...
化学式: Rb8 Tl4 Hg4 I24
-> [Step 0] 预处理: 原子数 40
-> [Step 2] Phase 1: 标准拓扑扫描...


Processing CIFs: 100%|██████████| 2240/2240 [45:30<00:00,  1.22s/it]

-> [Step 3] Phase 2: 检查孤立原子 (强制半径 4.0 Å)...
-> [Step 4] Phase 3: 全局对称化...
-> [Step 5] 生成特征...


In [4]:
print("\n处理完成！")
print(f"成功: {len(all_results)}, 失败: {len(failed_files)}")

if failed_files:
    print("失败文件示例:", failed_files[:5])
    # 保存错误日志
    with open("failed_log_1.txt", "w") as f:
        for line in failed_files:
            f.write(line + "\n")

if all_results:
    df = pd.DataFrame(all_results)
    output_path = 'features.csv'
    df.to_csv(output_path, index=False)
    print(f"特征表已保存至: {output_path}")
    print(f"特征维度: {df.shape}")
else:
    print("没有提取到任何数据。")


处理完成！
成功: 835, 失败: 1405
失败文件示例: ["mp-605034.cif: 'F'", "mp-14088.cif: 'F'", "mp-504729.cif: 'F'", "mp-557397.cif: 'F'", "mp-558111.cif: 'F'"]
特征表已保存至: features.csv
特征维度: (835, 283)
